## Library

In [1]:
import time
from dataretrieval import nwis
import pandas as pd
from IPython.display import clear_output
import geopandas as gpd
import glob
import os
import datetime
import pathlib
import earthaccess
import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np

/work/kwanhyuckkim_umass_edu/.conda/envs/swot-urban/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download Gage Height (00065) for SWOT Version D from 2025/04/01 with Merritt's Gage Info

In [2]:
# ============================================================================
# 1. Load target sites (USGS gages linked to SWORD)
# ============================================================================
usgs_link_merritt = pd.read_csv(
    "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv", 
    dtype={'gage_id': object}
)

target_sites = usgs_link_merritt[
    usgs_link_merritt.gage_agency == "USGS"
].gage_id.drop_duplicates().tolist()

print(f"Total target sites: {len(target_sites)}")
print(f"First 5 sites: {target_sites[:5]}")

Total target sites: 1723
First 5 sites: ['01010000', '01010500', '01011000', '01013500', '01014000']


/tmp/ipykernel_3469250/1816538775.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  usgs_link_merritt = pd.read_csv(


In [3]:
# ============================================================================
# 2. Download site information (datum, location, etc.)
# ============================================================================
print("\n" + "="*80)
print("Downloading site information...")
print("="*80)

site_info_parts = []
batch_size = 100  # NWIS API limit per request

for i in range(0, len(target_sites), batch_size):
    batch = target_sites[i:i+batch_size]
    clear_output(wait=True)
    print(f"Downloading site info: {i+1}-{min(i+batch_size, len(target_sites))} of {len(target_sites)}")
    
    try:
        # Get site information - returns tuple (df, md)
        result = nwis.get_info(sites=batch)
        
        # Handle different return types
        if isinstance(result, tuple):
            site_batch = result[0]  # First element is the dataframe
        else:
            site_batch = result
        
        # Check if we got valid data
        if site_batch is not None and isinstance(site_batch, pd.DataFrame) and not site_batch.empty:
            site_info_parts.append(site_batch)
            print(f"  → Downloaded: {len(site_batch)} sites")
        else:
            print(f"  → No data in this batch")
            
    except Exception as e:
        print(f"  → Error in batch {i}: {str(e)}")
        continue
    
    time.sleep(0.5)

# Combine all site info
if site_info_parts:
    all_site_info = pd.concat(site_info_parts, axis=0, ignore_index=False)
    print(f"\nTotal sites with info: {len(all_site_info)}")
else:
    print("\nWarning: No site information downloaded!")
    all_site_info = pd.DataFrame()

  → Downloaded: 23 sites

Total sites with info: 1724


In [ ]:
# ============================================================================
# 3. Download WSE time series data
# ============================================================================
print("\n" + "="*80)
print("Downloading WSE time series...")
print("="*80)

START, END = "2025-04-01", "2025-10-31"

all_wse_parts = []
failed_sites = []
success_count = 0

for idx, site in enumerate(target_sites, 1):
    clear_output(wait=True)
    print(f"Processing site {idx}/{len(target_sites)}: {site}")
    print(f"Success so far: {success_count}")
    
    try:
        # Download instantaneous values (15-min interval)
        # Parameter codes: 62617 (NAVD88, meters), 62622 (NAVD88, meters)
        result = nwis.get_iv(
            sites=site,
            startDT=START, 
            endDT=END,
            parameterCd="00065"
        )
        
        # Handle tuple return
        if isinstance(result, tuple):
            df = result[0]
        else:
            df = result
        
        if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
            # Reset index to get datetime as column
            df = df.reset_index()
            all_wse_parts.append(df)
            success_count += 1
            print(f"  → Success: {len(df)} records")
        else:
            print(f"  → No data available")
            failed_sites.append(site)
            
    except Exception as e:
        print(f"  → Error: {str(e)}")
        failed_sites.append(site)
        continue
    
    time.sleep(0.3)  # Rate limiting

Processing site 273/1723: 02312500
Success so far: 271


In [ ]:
# ============================================================================
# 4. Process and combine gage height data
# ============================================================================
if not all_wse_parts:
    print("\n ERROR: No gage height data downloaded!")
    raise SystemExit

all_gage = pd.concat(all_wse_parts, axis=0, ignore_index=True)

# Rename datetime column
if 'datetime' in all_gage.columns:
    all_gage = all_gage.rename(columns={'datetime': 'measurement_dt'})

all_gage['measurement_dt'] = pd.to_datetime(all_gage['measurement_dt'])

# Find the gage height value column (usually named like '00065' or '00065_Mean')
gage_col = [col for col in all_gage.columns if '00065' in col and 'cd' not in col.lower()][0]
all_gage = all_gage.rename(columns={gage_col: 'gage_height_ft'})

print("\n" + "="*80)
print("GAGE HEIGHT DATA SUMMARY")
print("="*80)
print(f"Total records: {len(all_gage):,}")
print(f"Unique sites: {all_gage['site_no'].nunique()}")
print(f"Date range: {all_gage['measurement_dt'].min()} to {all_gage['measurement_dt'].max()}")
print(f"Sites with data: {success_count}")
print(f"Failed sites: {len(failed_sites)}")

In [ ]:
import pandas as pd
import numpy as np

print("="*80)
print("Missing Gage Height Data Restructuring")
print("="*80)

# all_gage is assumed to be loaded in memory
# all_gage_miss = all_gage.copy()

# Step 1: Identify 00065_* columns
print("\nStep 1: Identifying 00065_* columns...")

# Find all columns starting with 00065
all_columns = all_gage.columns.tolist()
gage_height_columns = [col for col in all_columns if col.startswith('00065_')]

print(f"Total 00065_* columns found: {len(gage_height_columns)}")

# Step 2: Exclude datum-related columns
print("\nStep 2: Filtering out datum-related columns...")

# datum_keywords = [
#     'navd88', 'navd 1988', 'igld85', 'igld 85', 
#     'nws datum', 'usgs/nws datum',
#     'elevation referenced to'
# ]
datum_keywords = []

columns_to_exclude = []
for col in gage_height_columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in datum_keywords):
        columns_to_exclude.append(col)

print(f"Columns to exclude (datum-related): {len(columns_to_exclude)}")
for col in sorted(columns_to_exclude):
    print(f"  - {col}")

# Exclude columns and keep the remaining for processing
working_columns = [col for col in gage_height_columns if col not in columns_to_exclude]
print(f"\nWorking with {len(working_columns)} columns after exclusion")

# Step 3: Identify paired columns (upstream/downstream, headwater/tailwater, etc.)
print("\nStep 3: Identifying paired columns (upstream/downstream, headwater/tailwater)...")

pair_patterns = {
    'upstream_downstream': ['upstream', 'downstream', 'down stream'],
    'headwater_tailwater': ['headwater', 'tailwater'],
    'lock_chamber': ['lock chamber', 'lock level'],
}

paired_columns = {
    'upstream_downstream': {'left': [], 'right': []},
    'headwater_tailwater': {'left': [], 'right': []},
}

unpaired_columns = []

for col in working_columns:
    col_lower = col.lower()
    # upstream/downstream check
    if 'upstream' in col_lower:
        paired_columns['upstream_downstream']['left'].append(col)
    elif 'downstream' in col_lower or 'down stream' in col_lower:
        paired_columns['upstream_downstream']['right'].append(col)
    # headwater/tailwater check
    elif 'headwater' in col_lower:
        paired_columns['headwater_tailwater']['left'].append(col)
    elif 'tailwater' in col_lower:
        paired_columns['headwater_tailwater']['right'].append(col)
    else:
        unpaired_columns.append(col)

print(f"\nPaired columns found:")
for pair_type, cols in paired_columns.items():
    print(f"  {pair_type}:")
    print(f"    Left ({len(cols['left'])}): {cols['left']}")
    print(f"    Right ({len(cols['right'])}): {cols['right']}")

print(f"\nUnpaired columns ({len(unpaired_columns)}): will go to gage_height_missing")

# Step 4: Create new dataframe for restructuring
print("\nStep 4: Creating restructured dataframe...")

# Keep only essential columns
all_gage_miss = all_gage[['measurement_dt', 'site_no', 'gage_height_ft']].copy()

# By default, keep all rows and add new columns
all_gage_miss['gage_height_missing'] = np.nan
all_gage_miss['gage_height_missing_left'] = np.nan
all_gage_miss['gage_height_missing_right'] = np.nan
all_gage_miss['source_column'] = ''

print("Initial dataframe created with base columns")

# Step 5: Consolidate unpaired columns into gage_height_missing
print("\nStep 5: Consolidating unpaired columns into gage_height_missing...")

for col in unpaired_columns:
    mask = all_gage[col].notna()
    if mask.sum() > 0:
        no_value_yet = all_gage_miss['gage_height_missing'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            all_gage_miss.loc[update_mask, 'gage_height_missing'] = all_gage.loc[update_mask, col]
            all_gage_miss.loc[update_mask, 'source_column'] = col
            print(f"  {col}: {update_mask.sum():,} values")

# Step 6: Process paired columns: left/right separation
print("\nStep 6: Processing paired columns...")

# Process upstream/downstream
for left_col in paired_columns['upstream_downstream']['left']:
    mask = all_gage[left_col].notna()
    if mask.sum() > 0:
        no_value_yet = all_gage_miss['gage_height_missing_left'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            all_gage_miss.loc[update_mask, 'gage_height_missing_left'] = all_gage.loc[update_mask, left_col]
            for idx in all_gage_miss[update_mask].index:
                existing = all_gage_miss.loc[idx, 'source_column']
                new_source = f"left:{left_col}"
                all_gage_miss.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
            print(f"  LEFT - {left_col}: {update_mask.sum():,} values")

for right_col in paired_columns['upstream_downstream']['right']:
    mask = all_gage[right_col].notna()
    if mask.sum() > 0:
        no_value_yet = all_gage_miss['gage_height_missing_right'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            all_gage_miss.loc[update_mask, 'gage_height_missing_right'] = all_gage.loc[update_mask, right_col]
            for idx in all_gage_miss[update_mask].index:
                existing = all_gage_miss.loc[idx, 'source_column']
                new_source = f"right:{right_col}"
                all_gage_miss.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
            print(f"  RIGHT - {right_col}: {update_mask.sum():,} values")

# Process headwater/tailwater
for left_col in paired_columns['headwater_tailwater']['left']:
    mask = all_gage[left_col].notna()
    if mask.sum() > 0:
        no_value_yet = all_gage_miss['gage_height_missing_left'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            all_gage_miss.loc[update_mask, 'gage_height_missing_left'] = all_gage.loc[update_mask, left_col]
            for idx in all_gage_miss[update_mask].index:
                existing = all_gage_miss.loc[idx, 'source_column']
                new_source = f"left:{left_col}"
                all_gage_miss.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
            print(f"  LEFT - {left_col}: {update_mask.sum():,} values")

for right_col in paired_columns['headwater_tailwater']['right']:
    mask = all_gage[right_col].notna()
    if mask.sum() > 0:
        no_value_yet = all_gage_miss['gage_height_missing_right'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            all_gage_miss.loc[update_mask, 'gage_height_missing_right'] = all_gage.loc[update_mask, right_col]
            for idx in all_gage_miss[update_mask].index:
                existing = all_gage_miss.loc[idx, 'source_column']
                new_source = f"right:{right_col}"
                all_gage_miss.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
            print(f"  RIGHT - {right_col}: {update_mask.sum():,} values")

# Step 7: Output summary
print("\n" + "="*80)
print("RESTRUCTURING COMPLETE")
print("="*80)

print(f"\nDataframe shape: {all_gage_miss.shape}")
print(f"Columns: {list(all_gage_miss.columns)}")

print("\nValue counts per column:")
print(f"  gage_height_ft not null: {all_gage_miss['gage_height_ft'].notna().sum():,}")
print(f"  gage_height_missing not null: {all_gage_miss['gage_height_missing'].notna().sum():,}")
print(f"  gage_height_missing_left not null: {all_gage_miss['gage_height_missing_left'].notna().sum():,}")
print(f"  gage_height_missing_right not null: {all_gage_miss['gage_height_missing_right'].notna().sum():,}")
print(f"  source_column not empty: {(all_gage_miss['source_column'] != '').sum():,}")

# Stats for rows with missing gage_height but alternative available
missing_primary = all_gage_miss['gage_height_ft'].isna()
has_alternative = (
    all_gage_miss['gage_height_missing'].notna() | 
    all_gage_miss['gage_height_missing_left'].notna() | 
    all_gage_miss['gage_height_missing_right'].notna()
)
recoverable = missing_primary & has_alternative

print(f"\nRecoverable missing data:")
print(f"  gage_height_ft is NaN: {missing_primary.sum():,}")
print(f"  Has alternative sensor data: {has_alternative.sum():,}")
print(f"  Recoverable (NaN but has alternative): {recoverable.sum():,}")
print(f"  Recovery rate: {recoverable.sum() / missing_primary.sum() * 100:.2f}%")

# Source column statistics
print("\nTop 10 source columns:")
source_counts = all_gage_miss[all_gage_miss['source_column'] != '']['source_column'].value_counts().head(10)
for source, count in source_counts.items():
    print(f"  {source}: {count:,}")

print("\nSample of restructured data:")
print(all_gage_miss[recoverable].head(10))

all_gage_miss.to_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_gage_miss_restructured.parquet')
print("\nRestructured dataframe 'all_gage_miss' is ready for use!")

In [ ]:
# Convert all feet-based gage height columns to meters and rename columns
def feet_to_meter(series):
    return series * 0.3048

convert_cols = [
    'gage_height_ft',
    'gage_height_missing',
    'gage_height_missing_left',
    'gage_height_missing_right'
]

for col in convert_cols:
    meter_col = col.replace('_ft', '_m').replace('gage_height', 'gage_height_m')
    # If column exists and has any data
    if col in all_gage_miss.columns:
        # Generate converted column (meter)
        all_gage_miss[meter_col] = feet_to_meter(all_gage_miss[col])
        print(f"Converted {col} to meter ({meter_col})")

# You can also rename columns for clarity (optional, comment out if unnecessary)
rename_dict = {col: col.replace('_ft', '_orig_ft') for col in convert_cols if col in all_gage_miss.columns}
all_gage_miss.rename(columns=rename_dict, inplace=True)
print("Renamed original feet columns for clarity.")

print("\nColumns after conversion and renaming:")
print(list(all_gage_miss.columns))

all_gage_miss.to_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_gage_miss_restructured_meter.parquet')
print("\nRestructured dataframe 'all_gage_miss' is ready for use!")

#### CONVERT GAGE HEIGHT TO SWOT-COMPATIBLE WSE

In [ ]:
# ============================================================================
# 5. CONVERT GAGE HEIGHT TO SWOT-COMPATIBLE WSE 
# Following Harlan et al. (2025) methodology
# ============================================================================
print("\n" + "="*80)
print("CONVERTING TO SWOT-COMPATIBLE WSE")
print("Following Harlan et al. (2025) Paper Methodology")
print("="*80)

import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================================
# Step 0: Prepare data and merge with site info
# ============================================================================
print("\n--- Step 0: Preparing data ---")

if not all_site_info.empty:
    # Prepare site info with needed columns
    site_info_subset = all_site_info[['alt_va', 'alt_datum_cd', 'dec_lat_va', 'dec_long_va']].copy()
    site_info_subset = site_info_subset.reset_index()
    site_info_subset = site_info_subset.rename(columns={'index': 'site_no'})
    
    # Merge
    all_gage = all_gage.merge(site_info_subset, on='site_no', how='left')
    
    print(f"Sites with datum info: {all_gage['alt_va'].notna().sum():,}")
    print(f"Sites with coordinates: {all_gage['dec_lat_va'].notna().sum():,}")
else:
    print("Warning: No site info available for datum conversion")
    raise ValueError("Site info is required for datum conversion!")

In [ ]:
# -----------------------------------------------------------------------
# Step 1: Convert Gage Height to Absolute Elevation (NAVD88)
# -----------------------------------------------------------------------
print("\n--- Step 1: Gage Height → WSE (NAVD88) ---")

# Formula: WSE (NAVD88, ft) = Gage Height (ft) + Gage Datum Elevation (ft)
all_gage['wse_navd88_ft'] = all_gage['gage_height_ft'] + all_gage['alt_va']

# Convert feet to meters
all_gage['wse_navd88_m'] = all_gage['wse_navd88_ft'] * 0.3048

print(f"✓ Converted {all_gage['wse_navd88_m'].notna().sum():,} records to NAVD88")

In [ ]:
# -----------------------------------------------------------------------
# Step 2: NAVD88 → EGM2008 (tide-free) using VDatum transformation
# -----------------------------------------------------------------------
print("\n--- Step 2: NAVD88 → EGM2008 (tide-free) ---")
print("Note: This requires VDatum API or pre-computed geoid heights")
print("For now, we'll create a placeholder column for geoid offset")

# TODO: Implement actual VDatum API calls or use pre-computed geoid grid
# For demonstration, we'll add a placeholder column
# In practice, you would:
# 1. Use NOAA VDatum API: https://vdatum.noaa.gov/vdatumweb/api
# 2. Or use a pre-downloaded geoid height grid

# Placeholder - each site needs its own geoid offset
all_gage['geoid_offset_m'] = np.nan  # To be filled with actual VDatum values

# For now, let's create a function to compute this (you'll need to implement the actual API call)
def get_geoid_offset_placeholder(lat, lon):
    """
    Placeholder for VDatum transformation
    In practice, call VDatum API or use geoid grid
    
    Returns approximate NAVD88 → EGM2008 offset in meters
    """
    # This is just a rough approximation for demonstration
    # Actual values vary by location (-50m to +50m in US)
    # YOU MUST REPLACE THIS WITH ACTUAL VDATUM DATA
    return np.nan

# Apply to each unique site (more efficient than row-by-row)
unique_sites = all_gage[['site_no', 'dec_lat_va', 'dec_long_va']].drop_duplicates()
unique_sites['geoid_offset_m'] = unique_sites.apply(
    lambda row: get_geoid_offset_placeholder(row['dec_lat_va'], row['dec_long_va']), 
    axis=1
)

# Merge back
all_gage = all_gage.drop('geoid_offset_m', axis=1)
all_gage = all_gage.merge(unique_sites[['site_no', 'geoid_offset_m']], on='site_no', how='left')

# Apply transformation (when geoid offset is available)
all_gage['wse_egm2008_tide_free_m'] = all_gage['wse_navd88_m'] + all_gage['geoid_offset_m']

print("VDatum transformation not yet implemented")
print("   You need to:")
print("   1. Use NOAA VDatum API, OR")
print("   2. Download geoid height grid, OR")
print("   3. Use the transformation equations from the paper")

In [ ]:
# -----------------------------------------------------------------------
# Step 3: Tide-free → Mean-tide correction (Harlan et al. 2025, Eq. 11.4-11.5)
# -----------------------------------------------------------------------
print("\n--- Step 3: Tide-free → Mean-tide correction ---")

# Formula from SWOT User Handbook (JPL D-109532, 2025):
# correction = -0.099 × (1.5 × sin²(latitude) - 0.5) [meters]

all_gage['latitude_rad'] = np.radians(all_gage['dec_lat_va'])
all_gage['mean_tide_correction_m'] = -0.099 * (
    1.5 * np.sin(all_gage['latitude_rad'])**2 - 0.5
)

# Final WSE (SWOT-compatible)
all_gage['wse_swot_compatible_m'] = (
    all_gage['wse_egm2008_tide_free_m'] + all_gage['mean_tide_correction_m']
)

print(f"Applied mean-tide correction")
print(f"   Correction range: {all_gage['mean_tide_correction_m'].min():.4f} to {all_gage['mean_tide_correction_m'].max():.4f} m")

In [ ]:
# -----------------------------------------------------------------------
# Alternative: Relative comparison (median-normalized)
# -----------------------------------------------------------------------
print("\n--- Alternative: Relative (median-normalized) comparison ---")
print("(This is what the paper primarily used)")

# For each site, normalize by median
all_gage['gage_height_relative_m'] = all_gage.groupby('site_no')['gage_height_ft'].transform(
    lambda x: (x - x.median()) * 0.3048  # Convert to meters and normalize
)

print(f"✓ Created relative (median-normalized) gage height")
print("   This can be directly compared to median-normalized SWOT WSE")


In [ ]:
# ============================================================================
# 6. Summary and save
# ============================================================================
print("\n" + "="*80)
print("FINAL DATA SUMMARY")
print("="*80)

# Select final columns to keep
final_cols = [
    'site_no', 'measurement_dt', 
    'gage_height_ft',  # Original gage height
    'gage_height_relative_m',  # Relative (for paper's method)
    'wse_navd88_m',  # Absolute WSE (NAVD88)
    'wse_swot_compatible_m',  # Final (needs VDatum implementation)
    'mean_tide_correction_m',
    'alt_va', 'alt_datum_cd',  # Gage datum info
    'dec_lat_va', 'dec_long_va'  # Location
]

# Keep only existing columns
final_cols = [col for col in final_cols if col in all_gage.columns]
all_gage_final = all_gage[final_cols].copy()

print(f"Total records: {len(all_gage_final):,}")
print(f"Unique sites: {all_gage_final['site_no'].nunique()}")
print(f"\nColumns available:")
for col in final_cols:
    print(f"  - {col}")

In [ ]:
# ============================================================================
# 7. Save processed data
# ============================================================================
base_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_gages/"

# Save main data
output_path = base_path + "usgs_wse_processed_25.parquet"
all_gage_final.to_parquet(output_path, index=False)
print(f"\n✓ Processed data saved to: {output_path}")

# Save site info
if not all_site_info.empty:
    site_info_path = base_path + "usgs_site_info_25.parquet"
    all_site_info.to_parquet(site_info_path, index=True)
    print(f"✓ Site info saved to: {site_info_path}")

# Save failed sites
if failed_sites:
    failed_path = base_path + "failed_sites_25.csv"
    pd.DataFrame({'site_no': failed_sites}).to_csv(failed_path, index=False)
    print(f"✓ Failed sites saved to: {failed_path}")

In [ ]:
# ============================================================================
# 8. Display sample processed data
# ============================================================================
print("\n" + "="*80)
print("SAMPLE PROCESSED DATA")
print("="*80)
print(all_gage_final.head(10))

print("\n" + "="*80)
print("⭐ NEXT STEPS FOR SWOT COMPARISON ⭐")
print("="*80)
print("""
1. COMPLETE VDatum transformation:
   - Implement actual VDatum API calls OR
   - Download and use geoid height grid
   - Fill in 'geoid_offset_m' column

2. RECOMMENDED: Use RELATIVE comparison (paper's main method):
   - Use 'gage_height_relative_m' column
   - Compare with median-normalized SWOT WSE
   - This avoids datum conversion uncertainties

3. For ABSOLUTE comparison:
   - Use 'wse_swot_compatible_m' after VDatum implementation
   - Compare directly with SWOT WSE

4. Time matching:
   - Match SWOT overpass times with nearest ±1 hour USGS measurement
   - Use 'measurement_dt' column

The data is now ready for SWOT comparison using the relative method!
""")

print("="*80)
print("✓ PROCESSING COMPLETE!")
print("="*80)

## Merge buffered polygons with usgs gage heights

In [2]:
# Read the parquet file with NLCD data
buffered_polygons = gpd.read_parquet(
    "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/buffered_polygons_with_nlcd.parquet"
)

# Read the parquet file with usgs gage heights
usgs_height= pd.read_parquet("/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_gages/usgs_gage_height_23_25.parquet")

In [3]:
# NLCD sum/mean pairs
for year in [21, 22, 23, 24]:
    sum_col = f'nlcd_{year}_sum'
    mean_col = f'nlcd_{year}_mean'
    buffered_polygons.loc[buffered_polygons[sum_col] == 0, mean_col] = 0

# fctimp count/mean pair
buffered_polygons.loc[buffered_polygons['fctimp_count'] == 0, 'fctimp_mean'] = 0

### Download SWOT Sword of Science River Discharge Products Version 1 and check if it has usgs reach_id and site_no

In [4]:
# Get the access
earthaccess.login()

# Search for one granule
granule_info = earthaccess.search_data(
    short_name="SWOT_L4_DAWG_SOS_DISCHARGE",
    temporal=("2024-01-01", "2025-11-03"),
    # count=1
)

print(f"Found {len(granule_info)} granule(s)")

if granule_info:
    # Download the priors file
    print("\nDownloading priors file...")
    files = earthaccess.download(
        granule_info,
        local_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/"
    )

Found 9 granule(s)



QUEUEING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 4913.60it/s]
PROCESSING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 218833.25it/s]
COLLECTING RESULTS | : 100%|██████████| 18/18 [00:00<00:00, 421773.59it/s]


In [5]:
import netCDF4 as nc
import numpy as np
import pandas as pd
import glob
import os

# Find the priors file
priors_dir = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/"
priors_files = glob.glob(os.path.join(priors_dir, "na*0726*priors*.nc"))

if not priors_files:
    print(f"No files found")
    exit(1)

priors_file = priors_files[0]
print(f"File: {os.path.basename(priors_file)}\n")

# Open the netCDF file
ds = nc.Dataset(priors_file, 'r')

# Check what groups are available
print("Groups in this file:")
for group_name in ds.groups.keys():
    print(f"  - {group_name}")

# Look at the USGS group
if 'USGS' in ds.groups: 
    usgs = ds.groups['USGS']
    
    # Check dimensions
    print("\nDimensions:")
    for dim_name, dim in usgs.dimensions.items():
        print(f"  {dim_name}: {len(dim)}")
    
    # Check what variables are in USGS group
    print("\nVariables in USGS group:")
    for var_name in usgs.variables.keys():
        var = usgs.variables[var_name]
        print(f"  {var_name}: {var.shape}, {var.dtype}")
    
    # Convert USGS data to DataFrame
    # Get the main variables
    reach_ids = usgs.variables['USGS_reach_id'][:]
    
    # USGS_id is stored as char array, need to convert to strings
    usgs_id_chars = usgs.variables['USGS_id'][:]
    usgs_site_ids = []
    for row in usgs_id_chars:
        site_id = ''.join([c.decode('utf-8') if isinstance(c, bytes) else c for c in row]).strip()
        usgs_site_ids.append(site_id)
    
    # Create DataFrame
    usgs_df = pd.DataFrame({'reach_id': reach_ids, 'usgs_site_no': usgs_site_ids})
    
    # Save as parquet
    output_path = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/usgs_sword_linkage.parquet"
    usgs_df.to_parquet(output_path, index=False)
    
    print(f"Saved to: {output_path}")
    print(f"Shape: {usgs_df.shape}")
ds.close()

File: na_sword_v16_SOS_unconstrained_0001_20240726T123358_priors.nc

Groups in this file:
  - reaches
  - nodes
  - model
  - gbpriors
  - USGS
  - WSC
  - MEFCCWP
  - SWOT_SHAQ

Dimensions:
  num_days: 16279
  num_months: 12
  probability: 20
  nchars: 100
  num_USGS_reaches: 1080

Variables in USGS group:
  num_days: (16279,), int32
  CAL: (1080,), int32
  USGS_reaches: (1080,), int32
  USGS_reach_id: (1080,), int64
  USGS_flow_duration_q: (1080, 20), float64
  USGS_max_q: (1080,), float64
  USGS_monthly_q: (1080, 12), float64
  USGS_mean_q: (1080,), float64
  USGS_min_q: (1080,), float64
  USGS_two_year_return_q: (1080,), float64
  USGS_id: (1080, 100), |S1
  USGS_q: (1080, 16279), float64
  USGS_qt: (1080, 16279), float64
Saved to: /work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/usgs_sword_linkage.parquet
Shape: (1080, 2)


### Link USGS gage heights and Buffered river width polygons

In [4]:
# Read the linkage file
usgs_link = pd.read_parquet("/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/usgs_sword_linkage.parquet")
usgs_link_ted = pd.read_csv("/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/usgs_matches.csv", dtype={'USGS_id': object})
usgs_link_merritt = pd.read_csv("/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv", dtype={'gage_id': object})

# Rename usgs_site_no to site_no
usgs_link = usgs_link.rename(columns={'usgs_site_no': 'site_no'})
usgs_link_ted = usgs_link.rename(columns={'USGS_id': 'site_no'})
usgs_link_merritt = usgs_link_merritt.rename(columns={'gage_id': 'site_no'})

print(f"\nusgs_link shape: {usgs_link_merritt.shape}")
print(f"buffered_polygons shape: {buffered_polygons.shape}")
print(f"usgs_height shape: {usgs_height.shape}")


usgs_link shape: (174765, 45)
buffered_polygons shape: (38478, 48)
usgs_height shape: (4604886, 305)


/tmp/ipykernel_4597/3540460071.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  usgs_link_merritt = pd.read_csv("/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv", dtype={'gage_id': object})


In [5]:
# Step 0: Convert keys to 'object' dtype for safe merging
buffered_polygons['reach_id'] = buffered_polygons['reach_id'].astype('object')
usgs_link_merritt['reach_id'] = usgs_link_merritt['reach_id'].astype('object')
usgs_link_merritt['site_no'] = usgs_link_merritt['site_no'].astype('object')
usgs_height['site_no'] = usgs_height['site_no'].astype('object')

# Step 1: Merge buffered_polygons + usgs_link_merritt on 'reach_id'
unique_reach_in_polygons = set(buffered_polygons['reach_id'].dropna())
unique_reach_in_link = set(usgs_link_merritt['reach_id'].dropna())
overlap_reach = unique_reach_in_polygons & unique_reach_in_link

print(f"Number of overlapping reach_id between buffered_polygons and usgs_link_merritt: {len(overlap_reach)}")

pol_link = buffered_polygons.merge(usgs_link_merritt, on='reach_id', how='inner')
print(f"buffered_polygons + usgs_link_merritt merged dataframe shape: {pol_link.shape}")

Number of overlapping reach_id between buffered_polygons and usgs_link_merritt: 1425
buffered_polygons + usgs_link_merritt merged dataframe shape: (122112, 92)


In [6]:
pol_link

,x,y,reach_id,reach_len,n_nodes,wse_x,wse_var,width_x,width_var,facc,...,p_n_ch_mod,p_dam_id,p_n_nodes,p_low_slp,xtrk_dist,sword_version,collection_shortname,cycle_id,pass_id,crid
0,-96.248174,58.896461,71140100161,10193.837974,51,132.400009,1.975216,416.0,35756.299535,48136.312294,...,1,0,51,0,7421.99512,16,SWOT_L2_HR_RiverSP_2.0,5,121,PGC0
1,-96.248174,58.896461,71140100161,10193.837974,51,132.400009,1.975216,416.0,35756.299535,48136.312294,...,1,0,51,0,65877.93750,16,SWOT_L2_HR_RiverSP_2.0,5,160,PGC0
2,-96.248174,58.896461,71140100161,10193.837974,51,132.400009,1.975216,416.0,35756.299535,48136.312294,...,1,0,51,0,-62205.30273,16,SWOT_L2_HR_RiverSP_2.0,5,188,PGC0
3,-96.248174,58.896461,71140100161,10193.837974,51,132.400009,1.975216,416.0,35756.299535,48136.312294,...,1,0,51,0,-57671.61719,16,SWOT_L2_HR_RiverSP_2.0,5,399,PGC0
4,-96.248174,58.896461,71140100161,10193.837974,51,132.400009,1.975216,416.0,35756.299535,48136.312294,...,1,0,51,0,1540.37256,16,SWOT_L2_HR_RiverSP_2.0,5,466,PGC0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122107,-123.590430,72.485306,84100200071,19038.075631,95,47.500000,18.113083,48.0,273.896116,3661.105376,...,1,0,95,0,-12580.39355,16,SWOT_L2_HR_RiverSP_2.0,31,375,PIC2
122108,-123.590430,72.485306,84100200071,19038.075631,95,47.500000,18.113083,48.0,273.896116,3661.105376,...,1,0,95,0,46415.73828,16,SWOT_L2_HR_RiverSP_2.0,31,403,PIC2
122109,-123.590430,72.485306,84100200071,19038.075631,95,47.500000,18.113083,48.0,273.896116,3661.105376,...,1,0,95,0,25738.32227,16,SWOT_L2_HR_RiverSP_2.0,31,522,PIC2
122110,-123.590430,72.485306,84100200071,19038.075631,95,47.500000,18.113083,48.0,273.896116,3661.105376,...,1,0,95,0,-42269.95703,16,SWOT_L2_HR_RiverSP_2.0,32,69,PIC2


In [ ]:
# Step 2: Merge with usgs_height on 'site_no'
unique_site_in_pol_link = set(pol_link['site_no'].dropna())
unique_site_in_height = set(usgs_height['site_no'].dropna())
overlap_site = unique_site_in_pol_link & unique_site_in_height

print(f"Number of overlapping site_no between pol_link and usgs_height: {len(overlap_site)}")

final_df = pol_link.merge(usgs_height, on='site_no', how='inner')
print(f"buffered_polygons + usgs_link_merritt + usgs_height merged dataframe shape: {final_df.shape}")

# Count unique (reach_id, site_no) pairings in final_df
unique_pairs = final_df[['reach_id', 'site_no']].drop_duplicates()
print(f"Number of identical reach_id-site_no pairings after all merges: {unique_pairs.shape[0]}")

# Save as parquet for later spatial mapping with GeoPandas (GeoDataFrame)
output_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_sword_matchups_merritt.parquet"
final_df.to_parquet(output_path, index=False)
print(f"Saved to: {output_path}")

Number of overlapping site_no between pol_link and usgs_height: 501


#### Troubleshooting for reach id with SWORD reaches

In [15]:
# Load raw_reaches
raw_reaches = gpd.read_file("/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp")

# Check how many unique reach_ids in each dataset
print(f"raw_reaches unique reach_ids: {raw_reaches['reach_id'].nunique()}")
print(f"buffered_polygons unique reach_ids: {buffered_polygons['reach_id'].nunique()}")
print(f"usgs_link unique reach_ids: {usgs_link['reach_id'].nunique()}")

# Create sets for comparison
raw_reach_ids = set(raw_reaches['reach_id'])
buffered_reach_ids = set(buffered_polygons['reach_id'])
usgs_reach_ids = set(usgs_link['reach_id'])

# Check overlaps
print(f"\nOverlaps:")
print(f"  raw_reaches ∩ usgs_link: {len(raw_reach_ids.intersection(usgs_reach_ids))}")
print(f"  buffered_polygons ∩ usgs_link: {len(buffered_reach_ids.intersection(usgs_reach_ids))}")
print(f"  raw_reaches ∩ buffered_polygons: {len(raw_reach_ids.intersection(buffered_reach_ids))}")

# Check what's missing
print(f"\nMissing reach_ids:")
print(f"  USGS reach_ids NOT in raw_reaches: {len(usgs_reach_ids - raw_reach_ids)}")
print(f"  USGS reach_ids NOT in buffered_polygons: {len(usgs_reach_ids - buffered_reach_ids)}")
print(f"  buffered NOT in raw_reaches: {len(buffered_reach_ids - raw_reach_ids)}")

raw_reaches unique reach_ids: 38478
buffered_polygons unique reach_ids: 38478
usgs_link unique reach_ids: 1057

Overlaps:
  raw_reaches ∩ usgs_link: 762
  buffered_polygons ∩ usgs_link: 762
  raw_reaches ∩ buffered_polygons: 38478

Missing reach_ids:
  USGS reach_ids NOT in raw_reaches: 295
  USGS reach_ids NOT in buffered_polygons: 295
  buffered NOT in raw_reaches: 0
